## Company Sales Brochure Generator


Create a product that can generate marketing brochures about a company

1. for prospective clients
2. for investors
3. for recruitment

In [20]:
# libraries
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

from web_scraper import fetch_website_contents, fetch_website_links

In [2]:
# loading the environment variables
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("API Keys not found")
elif not api_key.startswith('sk-proj-'):
    print("API key found but wrong format")
elif api_key.strip() != api_key:
    print("API key found but contain unnecessary space")
else:
    print("API Key found and in use")


MODEL = 'gpt-5-nano'
openai = OpenAI()

API Key found and in use


In [3]:
links = fetch_website_links("https://edwarddonner.com")

# https://irene-busah.github.io/portfolio/

# links

In [4]:
# Using the LLM to figure out relevant links

links_system_prompt = """
    You are provided with a list of links found on a website.
    You are able to decide which of the links would be most relevant to include in a brochure about the company,
    such as links to an about page, or a company page, or Careers/Jobs pages.
    You should respond in JSON as in this example:

    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "career page", "url": "https://another.full.url/careers"}
        ]
    }
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company,
        response with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):
    """

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)

    return user_prompt

print(get_links_user_prompt('https://edwarddonner.com'))


        Here is the list of links on the website https://edwarddonner.com -
        Please decide which of these are relevant web links for a brochure about the company,
        response with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):
    #wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-en

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content':links_system_prompt},
            {'role': 'user', 'content': get_links_user_prompt(url)}
        ],
        response_format={'type': 'json_object'}
    )

    result = response.choices[0].message.content

    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")

    return links

select_relevant_links('https://edwarddonner.com')

Found 7 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio / skills',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f'## Landing Page:\n\n{contents}\n## Relevant Links:\n'

    for link in relevant_links['links']:
        result += f'\n\n### Link: {link['type']}\n'
        result += fetch_website_contents(link['url'])
    return result


print(fetch_page_and_all_relevant_links('https://edwarddonner.com'))

Found 6 relevant links
## Landing Page:

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from 

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
        You are looking at a company called: {company_name}
        Here are the contents of its landing page and other relevant pages;
        use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found 14 relevant links


'\n        You are looking at a company called: HuggingFace\n        Here are the contents of its landing page and other relevant pages;\n        use this information to build a short brochure of the company in markdown without code blocks.\n\n\n    ## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\n×\nWe are happy to share our intention to join forces with\nNVIDIA\n.\nRead the announcement\nNEW\nMicroduck: A Tiny Robot for AI Builders 🦆\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nThe AI community building the future.\nThe platform where the machine learning community co

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("HuggingFace", "https://huggingface.co")

Found 14 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community and platform that is building the future of machine learning. As a collaborative hub for the ML ecosystem, Hugging Face empowers developers, researchers, and enterprises to create, discover, and share AI models, datasets, and applications across all modalities — text, image, video, audio, and even 3D.

Located at the crossroads of open source innovation and enterprise solutions, Hugging Face fosters a global community that drives rapid ML progress and innovation.

---

## What We Offer

### Community & Collaboration Platform  
- **Explore 2M+ Models:** Access a vast repository of open source and state-of-the-art AI models in NLP, vision, speech, and more.  
- **Browse 500K+ Datasets:** Find, share, and contribute to datasets suited for any ML task.  
- **Spaces:** Easily create and deploy ML-powered web apps and demos.  
- **Open Source Libraries:** Transformers, Diffusers, Tokenizers, Safetensors, and more — tools widely adopted by AI practitioners worldwide.

### Enterprise-Grade Solutions  
- **Hugging Face PRO:** Tailored team and enterprise plans with advanced platform features, including:  
  - Enterprise-grade security  
  - Single Sign-On support  
  - Priority dedicated support  
  - Audit logs and resource groups  
  - Private dataset viewers  
- **Inference Providers & Endpoints:** Simplify integrating models into applications with scalable API access and optimized GPU-powered endpoints.  
- **Storage Buckets:** AI-native object storage for seamless data handling.

---

## Our Customers & Partners

More than 50,000 organizations trust Hugging Face, ranging from startups to global tech leaders, including:

- **AI2 (Allen Institute for AI)**  
- **Meta AI**  
- **Amazon**  
- **Google**  
- **Intel**  
- **Microsoft**  
- **Grammarly**  
- **Writer**

Our platform powers a broad spectrum of industries and applications, from research labs and academic institutions to major enterprises.

---

## Company Culture

- **Open Source at Heart:** Hugging Face is committed to open collaboration and transparency. The company maintains popular community-driven projects like *Transformers*, *Diffusers*, *Tokenizers*, and many others, encouraging contributions from global developers.  
- **Inclusive & Collaborative:** The culture fosters community involvement through forums, a Discord channel, GitHub interactions, and a vibrant blog featuring AI research, ideas, and news.  
- **Innovation-Focused:** Hugging Face continuously pushes the boundaries of ML technologies, recently announcing a strategic collaboration with NVIDIA, emphasizing their commitment to growth and cutting-edge hardware integration.

---

## Careers at Hugging Face

Join a fast-growing company that is shaping the future of AI!

- Work alongside renowned ML researchers and engineers in a dynamic, inclusive, and innovation-driven environment.  
- Build tools and platforms used worldwide by tens of thousands of organizations and millions of users.  
- Contribute to open source projects impacting the AI community globally.  
- Competitive compensation with opportunities for professional development and growth.

Opportunities are available for software engineers, ML engineers, data scientists, research scientists, product managers, and more.

Find out more and apply at [huggingface.co/careers](https://huggingface.co/careers)

---

## Get Started with Hugging Face

- Explore AI models and datasets at [huggingface.co/models](https://huggingface.co/models)  
- Browse and deploy Spaces apps at [huggingface.co/spaces](https://huggingface.co/spaces)  
- Join the community on [Discord](https://discord.gg/huggingface) and [Forum](https://discuss.huggingface.co)  
- Build your AI portfolio with free and paid options from individual to enterprise scale.

---

## Connect With Us

**Website:** [https://huggingface.co](https://huggingface.co)  
**GitHub:** [https://github.com/huggingface](https://github.com/huggingface)  
**Twitter:** [@huggingface](https://twitter.com/huggingface)  
**LinkedIn:** [Hugging Face](https://www.linkedin.com/company/huggingface)  
**Discord:** Active community server for instant collaboration and support  

---

Hugging Face — Empowering the next generation of AI builders. Join us and build the future of machine learning, together.

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found 11 relevant links


# Hugging Face Brochure

---

## The AI Community Building the Future

Hugging Face is a global platform at the forefront of machine learning innovation, enabling collaboration and accelerating AI development across the community. The company is dedicated to building open-source tools and resources that empower developers, researchers, and enterprises to create state-of-the-art AI models and applications with ease.

---

## What We Offer

### A Collaborative Platform  
- Host and collaborate on **unlimited public models, datasets, and applications**  
- Explore and contribute to a rich ecosystem of **over 2 million models** and **500,000 datasets** spanning modalities like text, image, video, audio, and 3D  
- Build your personal machine learning portfolio and share your work with the world

### Open Source Excellence  
- Transformers: Cutting-edge AI models for PyTorch with a community of over 164,000 contributors  
- Diffusers: Leading diffusion models powering image and video generation  
- Tokenizers, Safetensors, PEFT, TRL, and more — comprehensive ML tooling for production and research  

### Enterprise and Professional Solutions  
- **Hugging Face PRO and Enterprise Support:** Advanced platform with enterprise-grade security, Single Sign-On, audit logs, and priority support  
- Inference Providers: Access 45,000+ models through a single unified API with no added fees  
- Optimized GPU Inference Endpoints for easy deployment and scaling at competitive pricing  
- Storage Buckets: AI-native object storage for seamless data management  

---

## Our Customers & Partners

Trusted by over **50,000 organizations** including industry leaders and innovators such as:

- **Meta AI**  
- **Google**  
- **Microsoft**  
- **Amazon**  
- **Intel**  
- **Grammarly**  

We are also excited to share our intention to **join forces with NVIDIA**, underlining our commitment to accelerating AI innovation.

---

## Company Culture

- A community-first mindset that thrives on open collaboration and inclusive innovation  
- Commitment to open-source principles, empowering everyone from hobbyists to enterprise teams  
- Rapid iteration and release cycles, encouraging continuous learning and sharing within the AI ecosystem  
- Vibrant community engagement through **Discord, forums, GitHub, and blogs**

---

## Careers

Join a passionate team driving the future of AI! Hugging Face offers exciting opportunities for engineers, researchers, product managers, and more who want to:

- Contribute to world-class open-source ML projects  
- Work closely with a diverse global community of AI builders  
- Innovate on cutting-edge AI technologies with impact at scale  
- Thrive in a culture of learning, openness, and collaboration  

Find current openings and start your journey at Hugging Face by visiting their **Careers** page.

---

## Connect With Us

- Visit: [huggingface.co](https://huggingface.co)  
- Join the conversation on **Discord**  
- Explore on **GitHub**  
- Follow on **Twitter** and **LinkedIn**  

---

## Hugging Face — Build AI Together

Whether you’re an individual researcher, an enterprise team, or a community builder, Hugging Face provides the tools, resources, and supportive ecosystem to accelerate your AI journey.

Explore models, datasets, and start collaborating today!  

**Sign Up Now and Be Part of the Future of AI.**